# ML-оптимизация 3D-печатного BLDC 12S14P

Модель обучается на 144 расчётах FEMM и прогнозирует электромагнитный момент по трём параметрам:

- воздушный зазор;
- толщина магнита;
- ток.

Цель — подобрать размеры ротора, обеспечивающие максимальный момент при заданных ограничениях изготовления.

In [ ]:
import io, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
print('Библиотеки загружены')

## 1. Загрузка FEMM-датасета

После запуска ячейки выберите файл `BLDC_ML_dataset.csv`.

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print('Размер:', df.shape)
display(df.head())

## 2. Проверка данных

In [ ]:
key = ['gap_mm', 'magnet_thickness_mm', 'current_A']
print('Пропущенных значений:', int(df.isna().sum().sum()))
print('Повторяющихся комбинаций:', int(df.duplicated(key).sum()))
print('Диапазон зазора:', df.gap_mm.min(), '—', df.gap_mm.max(), 'мм')
print('Толщины магнитов:', sorted(df.magnet_thickness_mm.unique()))
print('Значения тока:', sorted(df.current_A.unique()))
assert len(df) == 144 and not df.isna().any().any() and not df.duplicated(key).any()
print('Датасет корректен')

## 3. Сравнение алгоритмов

Разделение выполняется по группам зазора: модель проверяется на значениях зазора, которых не видела во время обучения. Это строже обычного случайного разделения.

In [ ]:
features = ['gap_mm', 'magnet_thickness_mm', 'current_A']
X = df[features]
y = df['torque_mNm']
groups = df['gap_mm']
cv = GroupKFold(n_splits=6)

models = {
    'Linear regression': Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
    'Polynomial degree 2': Pipeline([('poly', PolynomialFeatures(2, include_bias=False)), ('scale', StandardScaler()), ('model', Ridge(alpha=1e-5))]),
    'Polynomial degree 3': Pipeline([('poly', PolynomialFeatures(3, include_bias=False)), ('scale', StandardScaler()), ('model', Ridge(alpha=1e-5))]),
    'SVR (RBF)': Pipeline([('scale', StandardScaler()), ('model', SVR(C=100, epsilon=0.001))]),
    'Random Forest': RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=500, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, max_depth=2, learning_rate=0.04, loss='huber', random_state=42)
}

rows = []
cv_predictions = {}
for name, model in models.items():
    pred = cross_val_predict(model, X, y, cv=cv, groups=groups, n_jobs=-1)
    cv_predictions[name] = pred
    rows.append({
        'Model': name,
        'R²': r2_score(y, pred),
        'MAE, mN·m': mean_absolute_error(y, pred),
        'RMSE, mN·m': mean_squared_error(y, pred) ** 0.5
    })

metrics = pd.DataFrame(rows).sort_values('RMSE, mN·m').reset_index(drop=True)
display(metrics.style.format({'R²':'{:.6f}', 'MAE, mN·m':'{:.5f}', 'RMSE, mN·m':'{:.5f}'}))

## 4. Обучение лучшей модели и проверка

In [ ]:
best_name = metrics.loc[0, 'Model']
best_model = models[best_name].fit(X, y)
pred = cv_predictions[best_name]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y, pred, alpha=0.75)
lims = [min(y.min(), pred.min()), max(y.max(), pred.max())]
ax.plot(lims, lims, '--', color='black')
ax.set(xlabel='FEMM torque, mN·m', ylabel='ML prediction, mN·m', title=f'Best model: {best_name}')
plt.show()

print('Лучшая модель:', best_name)
display(metrics.iloc[[0]])

## 5. Проверка нашей 3D-модели

Исходные параметры: зазор 0,95 мм, магнит 2 мм, ток 2 А.

In [ ]:
prototype = pd.DataFrame([[0.95, 2.0, 2.0]], columns=features)
prediction = float(best_model.predict(prototype)[0])
actual = float(df.query('gap_mm == 0.95 and magnet_thickness_mm == 2 and current_A == 2').torque_mNm.iloc[0])
print(f'FEMM: {actual:.6f} mN·m')
print(f'ML:   {prediction:.6f} mN·m')
print(f'Ошибка: {abs(prediction-actual)/actual*100:.4f}%')

## 6. Подбор размеров под конкретный случай

Измените три ограничения ниже. Программа ищет лучший вариант только внутри диапазона FEMM-датасета.

In [ ]:
selected_current_A = 2.0
minimum_safe_gap_mm = 0.95
maximum_magnet_thickness_mm = 2.0

gaps = np.arange(max(0.50, minimum_safe_gap_mm), 1.5001, 0.005)
thicknesses = np.arange(1.50, min(3.00, maximum_magnet_thickness_mm) + 0.0001, 0.01)
search = pd.MultiIndex.from_product(
    [gaps, thicknesses, [selected_current_A]], names=features
).to_frame(index=False)
search['predicted_torque_mNm'] = best_model.predict(search[features])
opt = search.loc[search.predicted_torque_mNm.idxmax()].copy()

stator_OD = 38.50
magnet_inner_D = stator_OD + 2 * opt.gap_mm
magnet_seat_D = magnet_inner_D + 2 * opt.magnet_thickness_mm
rotor_OD = magnet_seat_D + 5.60

result = pd.DataFrame({
    'Parameter': ['Air gap', 'Magnet thickness', 'Current', 'Magnet inner diameter', 'Magnet seat diameter', 'Rotor outer diameter', 'Predicted torque'],
    'Value': [opt.gap_mm, opt.magnet_thickness_mm, opt.current_A, magnet_inner_D, magnet_seat_D, rotor_OD, opt.predicted_torque_mNm],
    'Unit': ['mm', 'mm', 'A', 'mm', 'mm', 'mm', 'mN·m']
})
display(result.style.format({'Value':'{:.3f}'}))

## 7. Сохранение модели и результатов

In [ ]:
joblib.dump(best_model, 'BLDC_torque_ML_model.joblib')
metrics.to_csv('BLDC_ML_metrics.csv', index=False)
result.to_csv('BLDC_recommended_dimensions.csv', index=False)
print('Файлы созданы. Скачайте их из панели Files слева.')